# Week 9 Lab 1: Text Classification via Embeddings (Keras)

> **Goal**: Implement a complete NLP pipeline—Tokenization, Word Embeddings, and CNN processing—to classify movie reviews as Positive or Negative using TensorFlow/Keras. At the end, we export the model for Edge deployment.

## 1. Setup Data

**Concepts**: We use the standard IMDb database of 50,000 movie reviews. Our goal is to predict sentiment strictly based on the text.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, datasets
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load IMDb data. We restrict vocabulary to the top 10,000 most common words.
vocab_size = 10000
(train_data, train_labels), (test_data, test_labels) = datasets.imdb.load_data(num_words=vocab_size)

print(f"Training reviews: {len(train_data)}")
print(f"First review sample (encoded): {train_data[0][:10]}...")

## 2. Preprocessing (Padding Sequences)

**Task**: Reviews have variable lengths. A Neural Network expects fixed-length tensors inside a batch. We use `pad_sequences` to ensure every review is exactly 200 words long (padding zeros at the end or truncating).

In [ ]:
maxlen = 200

train_padded = pad_sequences(train_data, maxlen=maxlen, padding='post', truncating='post')
test_padded = pad_sequences(test_data, maxlen=maxlen, padding='post', truncating='post')

print(f"Shape of padded training data: {train_padded.shape}")

## 3. Building the 1D TextCNN

**Theory**: We start with an `Embedding` layer that projects our discrete word IDs into dense 64-dimensional vectors. 
Next, we slide a 1D Convolution over the text to look for patterns like 'not good' or 'masterpiece'. Finally, we use Global Max Pooling to squash the sequence down to a single vector.

In [ ]:
model = models.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=64, input_length=maxlen),
    
    # Extract local N-gram features
    layers.Conv1D(128, 5, activation='relu'),
    layers.GlobalMaxPooling1D(),
    
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid') # Binary Output (0 or 1)
])

model.summary()

**Observation**: The Embedding block maps 10,000 words into 64 dimensions each. Thus, it contains 640,000 trainable parameters!

## 4. Compile and Train

Since this is a binary classification (Positive/Negative), we must use `binary_crossentropy`.

In [ ]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

history = model.fit(train_padded, train_labels, epochs=3, batch_size=64, 
                    validation_data=(test_padded, test_labels))

## 5. Export for Jetson Edge Deployment

We can deploy NLP to smart edge devices. We run a TensorFlow Lite conversion. This produces a micro-optimized model that runs beautifully on Nvidia Jetson chips without PyTorch overhead.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('sentiment_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("Model successfully exported to: sentiment_model.tflite")
print("Transfer this file and the vocabulary map to your Jetson device!")